# Pipeline de Entrenamiento del Modelo de Deep Learning

El propósito de este notebook es entrenar un modelo de clasificación de deep learning (una red neuronal) utilizando los datos procesados en el notebook anterior. El modelo se entrenará para predecir si un cliente suscribirá un depósito a plazo. Utilizaremos MLflow para registrar los parámetros del modelo, las métricas de entrenamiento y evaluación, y el modelo entrenado.

## Cargar Bibliotecas y Datos Procesados

In [ ]:
import pandas as pd
import numpy as np
import os
import mlflow
import mlflow.keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

In [ ]:
# Definir la ruta a los datos procesados
processed_data_dir = '../processed_data/'

# Nombres de los archivos
train_features_file = os.path.join(processed_data_dir, 'train_features.csv')
train_target_file = os.path.join(processed_data_dir, 'train_target.csv')
test_features_file = os.path.join(processed_data_dir, 'test_features.csv')
test_target_file = os.path.join(processed_data_dir, 'test_target.csv')

# Cargar los datos
try:
    X_train = pd.read_csv(train_features_file)
    y_train = pd.read_csv(train_target_file).squeeze() # .squeeze() para convertir DataFrame de una columna a Series
    X_test = pd.read_csv(test_features_file)
    y_test = pd.read_csv(test_target_file).squeeze()
    data_loaded_successfully = True
except FileNotFoundError as e:
    print(f"Error al cargar los datos: {e}")
    print("Asegúrese de que el notebook '02_feature_pipeline.ipynb' se haya ejecutado correctamente.")
    # Crear DataFrames vacíos para permitir que el resto del notebook se "ejecute" sin errores de variables no definidas
    X_train, y_train, X_test, y_test = pd.DataFrame(), pd.Series(dtype='float64'), pd.DataFrame(), pd.Series(dtype='float64')
    data_loaded_successfully = False

In [ ]:
if data_loaded_successfully:
    print("Formas de los datos cargados:")
    print(f"X_train: {X_train.shape}")
    print(f"y_train: {y_train.shape}")
    print(f"X_test: {X_test.shape}")
    print(f"y_test: {y_test.shape}")
    
    print("\nPrimeras filas de X_train:")
    display(X_train.head())
    print("\nPrimeras filas de y_train:")
    display(y_train.head())
else:
    print("No se pudieron cargar los datos. Las formas no se pueden mostrar.")

## Definición del Modelo de Red Neuronal (MLP)

Definiremos un Perceptrón Multicapa (MLP) utilizando la API Sequential de Keras. La arquitectura será la siguiente:

*   **Capa de Entrada:** Recibirá el número de características de nuestros datos de entrenamiento (`X_train.shape[1]`). Usará la función de activación ReLU.
*   **Capas Ocultas:** Se usarán una o dos capas ocultas, también con activación ReLU. ReLU (Rectified Linear Unit) es una función de activación común que ayuda a mitigar el problema del desvanecimiento del gradiente.
*   **Capas de Dropout (Opcional):** Se pueden añadir capas de Dropout después de las capas ocultas para reducir el sobreajuste (overfitting). Dropout desactiva aleatoriamente un porcentaje de neuronas durante el entrenamiento.
*   **Capa de Salida:** Tendrá una sola neurona con función de activación Sigmoide. La función Sigmoide produce una salida entre 0 y 1, adecuada para problemas de clasificación binaria, representando la probabilidad de pertenencia a la clase positiva.

In [ ]:
model = None
if data_loaded_successfully and not X_train.empty:
    input_dim = X_train.shape[1]
    
    model = Sequential([
        Dense(128, activation='relu', input_shape=(input_dim,)), # Capa de entrada y primera capa oculta
        Dropout(0.3), # Dropout para regularización
        Dense(64, activation='relu'), # Segunda capa oculta
        Dropout(0.3), # Dropout para regularización
        Dense(1, activation='sigmoid') # Capa de salida para clasificación binaria
    ])
else:
    print("No se pueden definir las dimensiones de entrada del modelo porque X_train está vacío o no se cargó.")

In [ ]:
if model:
    print(model.summary())

## Configuración de MLflow

MLflow se utilizará para el seguimiento de nuestros experimentos de machine learning. Esto incluye registrar parámetros, métricas, y artefactos como el modelo entrenado. Esto facilita la comparación entre diferentes ejecuciones y la reproducibilidad de los resultados.

In [ ]:
experiment_name = "Bank Marketing Classification"
mlflow.set_experiment(experiment_name)

print(f"MLflow experiment set to: '{experiment_name}'")

## Compilación y Entrenamiento del Modelo

**Compilación:**
Antes de entrenar el modelo, necesitamos configurarlo con:
*   **Optimizador:** Algoritmo para ajustar los pesos de la red (e.g., `Adam`, SGD). `Adam` es una elección popular y robusta.
*   **Función de Pérdida:** Mide qué tan bien se desempeña el modelo en los datos de entrenamiento (e.g., `binary_crossentropy` para clasificación binaria).
*   **Métricas:** Se utilizan para monitorear el proceso de entrenamiento y evaluación (e.g., `accuracy`). Calcularemos F1, Precision y Recall manualmente después del entrenamiento para tener un control más detallado sobre el logging en MLflow.

**Entrenamiento:**
El modelo se entrena utilizando el método `fit()`. Le proporcionaremos los datos de entrenamiento (`X_train`, `y_train`), el número de épocas (iteraciones sobre todo el conjunto de entrenamiento), y el tamaño del batch (número de muestras procesadas antes de actualizar los pesos). También incluiremos datos de validación (`X_test`, `y_test`) para monitorear el rendimiento del modelo en datos no vistos durante el entrenamiento y detectar sobreajuste.

In [ ]:
# Callback para registrar métricas en MLflow durante el entrenamiento
class MLflowMetricsCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        for key, value in logs.items():
            mlflow.log_metric(key, value, step=epoch)

if data_loaded_successfully and model is not None and not X_train.empty and not y_train.empty:
    with mlflow.start_run() as run:
        run_id = run.info.run_id
        print(f"MLflow Run ID: {run_id}")
        
        # Parámetros del modelo y entrenamiento
        learning_rate = 0.001
        epochs = 20 # Un número pequeño para ejecución rápida, aumentar para mejores resultados
        batch_size = 64
        optimizer_name = 'Adam'
        
        # Log de parámetros del modelo en MLflow
        mlflow.log_param("model_type", "MLP_Sequential")
        mlflow.log_param("hidden_layer_1_units", model.layers[0].units)
        mlflow.log_param("hidden_layer_1_activation", model.layers[0].activation.__name__)
        if len(model.layers) > 3: # Asumiendo Dense -> Dropout -> Dense -> Dropout -> Dense
             mlflow.log_param("hidden_layer_2_units", model.layers[2].units)
             mlflow.log_param("hidden_layer_2_activation", model.layers[2].activation.__name__)
             mlflow.log_param("dropout_1_rate", model.layers[1].rate)
             mlflow.log_param("dropout_2_rate", model.layers[3].rate)
        mlflow.log_param("output_layer_activation", model.layers[-1].activation.__name__)
        
        # Log de parámetros de entrenamiento en MLflow
        mlflow.log_param("optimizer", optimizer_name)
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("epochs", epochs)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("loss_function", "binary_crossentropy")

        # Compilación del modelo
        optimizer = Adam(learning_rate=learning_rate)
        model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
        print("\nModelo compilado.")

        # Entrenamiento del modelo
        print("Iniciando entrenamiento del modelo...")
        history = model.fit(
            X_train,
            y_train,
            epochs=epochs,
            batch_size=batch_size,
            validation_data=(X_test, y_test),
            callbacks=[MLflowMetricsCallback()], # Callback para MLflow
            verbose=1 # 0 = silent, 1 = progress bar, 2 = one line per epoch
        )
        print("Entrenamiento completado.")

        # El callback MLflowMetricsCallback ya registra 'loss', 'accuracy', 'val_loss', 'val_accuracy' por época.
        # Si no se usara el callback, se podría hacer así:
        # for epoch, acc in enumerate(history.history['accuracy']):
        #     mlflow.log_metric("accuracy", acc, step=epoch)
        # for epoch, val_acc in enumerate(history.history['val_accuracy']):
        #     mlflow.log_metric("val_accuracy", val_acc, step=epoch)
        # for epoch, loss_val in enumerate(history.history['loss']):
        #     mlflow.log_metric("loss", loss_val, step=epoch)
        # for epoch, val_loss_val in enumerate(history.history['val_loss']):
        #     mlflow.log_metric("val_loss", val_loss_val, step=epoch)
        
        # Guardar el modelo entrenado (se hace más adelante en la sección correspondiente)

else:
    print("No se puede compilar ni entrenar el modelo debido a que los datos no se cargaron o el modelo no se definió.")
    # Para evitar errores si el bloque anterior no se ejecuta, definimos run_id como None
    run_id = None 

## Evaluación del Modelo en el Conjunto de Prueba

Una vez entrenado el modelo, lo evaluaremos en el conjunto de prueba (`X_test`, `y_test`), que no fue utilizado durante el entrenamiento. Esto nos dará una estimación imparcial de su rendimiento.
Calcularemos las siguientes métricas:
*   **Accuracy:** Proporción de predicciones correctas.
*   **Precision:** De todas las predicciones positivas, cuántas fueron realmente positivas. Importante cuando los falsos positivos son costosos.
*   **Recall (Sensibilidad):** De todos los casos positivos reales, cuántos fueron identificados correctamente. Importante cuando los falsos negativos son costosos.
*   **F1-score:** Media armónica de Precision y Recall, útil para clases desbalanceadas.

In [ ]:
if data_loaded_successfully and model is not None and run_id is not None and not X_test.empty:
    # Asegurarnos de que estamos dentro del contexto de la ejecución de MLflow para loguear métricas
    # Si el bloque anterior falló, run_id podría ser None, o podríamos no estar en un 'with mlflow.start_run()'
    # Para este notebook, asumimos que si run_id existe, la ejecución de MLflow sigue activa desde el bloque anterior.
    # En un script separado, sería más robusto reabrir el run_id si es necesario.
    
    print("\nEvaluando el modelo en el conjunto de prueba...")
    y_pred_proba = model.predict(X_test)
    y_pred = (y_pred_proba > 0.5).astype(int) # Convertir probabilidades a clases binarias (0 o 1)
    
    # Calcular métricas
    test_accuracy = accuracy_score(y_test, y_pred)
    test_f1 = f1_score(y_test, y_pred)
    test_precision = precision_score(y_test, y_pred)
    test_recall = recall_score(y_test, y_pred)
    
    # Log de métricas de evaluación en MLflow
    mlflow.log_metric("test_accuracy", test_accuracy)
    mlflow.log_metric("test_f1_score", test_f1)
    mlflow.log_metric("test_precision", test_precision)
    mlflow.log_metric("test_recall", test_recall)
    
    print("\nMétricas de evaluación en el conjunto de prueba:")
    print(f"  Accuracy:  {test_accuracy:.4f}")
    print(f"  F1-score:  {test_f1:.4f}")
    print(f"  Precision: {test_precision:.4f}")
    print(f"  Recall:    {test_recall:.4f}")
    
    print("\nMatriz de Confusión:")
    cm = confusion_matrix(y_test, y_pred)
    print(cm)
    # Podríamos loguear la matriz de confusión también, por ejemplo, como una imagen o un artefacto JSON.
    # mlflow.log_dict(cm, "confusion_matrix.json") # No es un dict directamente, necesitaría conversión
else:
    print("No se puede evaluar el modelo. Asegúrese de que los datos se cargaron, el modelo se entrenó y MLflow run está activo.")

## Guardar el Modelo Entrenado

Después del entrenamiento y evaluación, guardaremos el modelo para su uso futuro (e.g., para realizar predicciones en nuevos datos o para desplegarlo en una aplicación).
El modelo se guardará localmente en el formato nativo de Keras (`.keras`) y también se registrará como un artefacto en MLflow, lo que permite un mejor versionado y gestión.

In [ ]:
if data_loaded_successfully and model is not None and run_id is not None:
    # Crear el directorio de modelos si no existe
    models_dir = "../models/"
    try:
        os.makedirs(models_dir, exist_ok=True)
        print(f"Directorio '{models_dir}' listo.")
    except OSError as e:
        print(f"Error al crear el directorio {models_dir}: {e}")
        # Aún así intentaremos guardar, ya que el directorio podría existir.

    # Guardar el modelo Keras localmente
    model_path = os.path.join(models_dir, "bank_marketing_model.keras")
    try:
        model.save(model_path)
        print(f"Modelo guardado localmente en: {model_path}")
    except Exception as e:
        print(f"Error al guardar el modelo localmente: {e}")
    
    # Loguear el modelo en MLflow
    # El modelo se guarda dentro del directorio de artefactos de la ejecución actual de MLflow.
    try:
        mlflow.keras.log_model(model, artifact_path="keras-model", registered_model_name="BankMarketingModelKeras")
        print("Modelo logueado en MLflow bajo el path 'keras-model' y registrado como 'BankMarketingModelKeras'.")
    except Exception as e:
        print(f"Error al loguear el modelo Keras en MLflow: {e}")
        
    # Finalizar la ejecución de MLflow explícitamente si no se usa 'with'
    # En este caso, 'with mlflow.start_run()' maneja el final de la ejecución.
    # mlflow.end_run() # No es necesario aquí debido al 'with' statement
else:
    print("No se puede guardar el modelo. Asegúrese de que el modelo se entrenó y MLflow run está activo.")

## Conclusiones del Entrenamiento

En este notebook, hemos entrenado un modelo de red neuronal para clasificar si un cliente de banco suscribirá un depósito a plazo. Se utilizó una arquitectura MLP con capas densas, funciones de activación ReLU y Sigmoide, y regularización con Dropout.

El proceso de entrenamiento y los resultados fueron registrados utilizando MLflow. Esto incluyó:
*   **Parámetros del Modelo:** Arquitectura de la red, optimizador, tasa de aprendizaje, etc.
*   **Métricas de Entrenamiento:** Pérdida (loss) y exactitud (accuracy) tanto en el conjunto de entrenamiento como en el de validación a lo largo de las épocas.
*   **Métricas de Evaluación Final:** Accuracy, F1-score, Precision y Recall en el conjunto de prueba.
*   **Modelo Entrenado:** El modelo final fue guardado localmente y también como un artefacto en MLflow para su posterior uso y versionado.

Las métricas obtenidas en el conjunto de prueba nos dan una idea del rendimiento del modelo en datos no vistos. Dependiendo de los valores de estas métricas (especialmente F1-score, Precision y Recall, dado el posible desbalance de clases), se podrían realizar ajustes adicionales, como probar diferentes arquitecturas de red, optimizar hiperparámetros, o aplicar técnicas más avanzadas de manejo de desbalance de clases si el rendimiento no es satisfactorio para los objetivos del negocio.

El modelo guardado está listo para ser utilizado en un pipeline de inferencia o desplegado como parte de una aplicación (por ejemplo, con Streamlit).